In [ ]:
"""
Beach wave collision simulation
================================
Model: 2D scalar wave equation implemented via a pseudodifferential
       operator (psipy), with anisotropy, bottom friction, and Coriolis.

Physical setup:
  Two water blades travel toward each other along the x-axis.
  Their collision generates a lateral wave propagating in the y direction,
  mimicking the swash/backwash interaction observed on gently sloping beaches.

Geometry:
  x ∈ [-Lx/2, Lx/2]  — cross-shore direction
  y ∈ [-Ly/2, Ly/2]  — longshore direction (periodic boundaries)

  Blade 1: starts at x = +X1, travels in -x direction (angle = 0, parallel to y)
  Blade 2: starts at x = -X2, travels in +x direction (angle = ALPHA_R, slightly tilted)

Equation solved:
  ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t

  where the principal symbol encodes wave propagation, anisotropy, and Coriolis:
  a(ξ) = C²·(A_XX·ξ² + 2·A_XY·ξη + A_YY·η²) + F_CORIOLIS·ξη

Physical effects:
  - Anisotropy (A_XX, A_YY, A_XY): different propagation speeds in x vs y
  - Coriolis (F_CORIOLIS): deflects wave energy sideways (Northern hemisphere: rightward)
  - Bottom friction (GAMMA): dissipates energy over time (blades decay realistically)

Parameter summary:
  C_SQUARED          wave speed squared (m²/s²)
  ALPHA_DEG          angle between the two blades (degrees)
  LAMBDA             blade characteristic wavelength, controls blade width (m)
  BLADE_WIDTH_RATIO  sigma = LAMBDA / BLADE_WIDTH_RATIO  (larger → narrower blade)
  HEAVISIDE_SHARPNESS  steepness of the blade leading edge  (larger → sharper front)
  SPEED_FACTOR       scales initial velocity  (1.0 = full WKB speed)
  GAMMA              bottom friction coefficient (s⁻¹)
  F_CORIOLIS         Coriolis parameter f = 2Ω·sin(latitude) (s⁻¹)
  A_XX, A_YY, A_XY   anisotropy matrix coefficients
"""

from solver import *
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML


In [ ]:
# ── 1. Physical and simulation parameters ────────────────────────────────────

# ── Wave speed ──
C_SQUARED = 1.0        # constant c² (m²/s²); replace with G*ALPHA*(x+X0) for variable speed

# ── Wave / blade properties ──
LAMBDA             = 4.0   # characteristic wavelength controlling blade width (m)
BLADE_WIDTH_RATIO  = 6.0   # sigma = LAMBDA / BLADE_WIDTH_RATIO  (larger → narrower)
HEAVISIDE_SHARPNESS = 10.0 # steepness of the blade leading edge  (larger → sharper front)
SPEED_FACTOR       = 0.5   # initial velocity scale  (1.0 = full WKB, <1 = slower)

# ── Blade geometry ──
ALPHA_DEG          = 1.0   # tilt angle of blade 2 relative to blade 1 (degrees)
ALPHA_R            = np.radians(ALPHA_DEG)
BLADE1_ANGLE       = 0.0   # blade 1: parallel to y-axis, travels in -x
BLADE2_ANGLE       = ALPHA_R  # blade 2: tilted by ALPHA_R, travels in +x

# ── Anisotropy matrix  A = [[A_XX, A_XY], [A_XY, A_YY]] ──
# symbol_wave = C² · (A_XX·ξ² + 2·A_XY·ξη + A_YY·η²)
# A_XX > A_YY : faster cross-shore than longshore propagation
# A_XY ≠ 0   : couples x and y directions
A_XX = 1.0   # cross-shore weight
A_YY = 1.0   # longshore weight  (set < 1 to slow down longshore propagation)
A_XY = 0.0   # off-diagonal coupling (0 = principal axes aligned with x, y)

# ── Dissipation and Coriolis ──
GAMMA      = 0.0    # bottom friction (s⁻¹); 0 = no decay, 0.05 = moderate decay
F_CORIOLIS = -0.0   # Coriolis parameter (s⁻¹); 0 = off, >0 = Northern hemisphere

# ── Velocity balance between the two blades ──
# Compensates for amplitude differences between straight and angled blades.
VELOCITY_SCALE_BLADE2 = 3


In [ ]:
# ── 2. Grid setup ────────────────────────────────────────────────────────────
Lx, Ly   = 10.0, 14.0
# Nx, Ny = 32, 32 
# Nx, Ny = 64, 64    
# Nx, Ny = 128, 128    
Nx, Ny = 256, 256
# Lt, Nt   = 20.0, 800
Lt, Nt   = 30.0, 1000
n_frames = 400

X1 =  (Lx) / 2.0
X2 = -(Lx) / 2.0

# Default 'xy' indexing: rows = y (axis=0), columns = x (axis=1)
# Consistent with psipy's internal convention and imshow's origin='lower'
xx, yy = np.meshgrid(np.linspace(-Lx/2, Lx/2, Nx),
                     np.linspace(-Ly/2, Ly/2, Ny),  indexing='ij')

In [ ]:
# ── 3. SymPy symbols and principal symbol ────────────────────────────────────

x, y, t  = sp.symbols('x y t', real=True)
xi, eta  = sp.symbols('xi eta', real=True)
u_func   = Function('u')
u        = u_func(t, x, y)

# Full principal symbol:
#   a(ξ) = C²·(A_XX·ξ² + 2·A_XY·ξη + A_YY·η²)   ← wave propagation + anisotropy
#          + F_CORIOLIS·ξη                          ← Coriolis deflection
symbol_wave     = C_SQUARED * (A_XX * xi**2 + 2*A_XY * xi*eta + A_YY * eta**2)
symbol_coriolis = F_CORIOLIS * xi * eta
symbol_num      = symbol_wave + symbol_coriolis

print("Principal symbol:")
print("  a(ξ) =", symbol_num)


In [ ]:
# ── 4. Wave equation ─────────────────────────────────────────────────────────
#
#   ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t
#
#   Term 1: -psiOp(a(ξ), u)  — wave propagation, anisotropy, Coriolis (in symbol)
#   Term 2: -GAMMA·∂u/∂t     — bottom friction (energy dissipation, not in symbol
#                               because it is a zeroth-order term in space)

gamma    = sp.Symbol('gamma', positive=True)
equation = sp.Eq(
    diff(u, t, 2),
    -psiOp(symbol_num, u) - gamma * diff(u, t)
)
equation_num = equation.subs({gamma: GAMMA})

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp(a(ξ), u) - {GAMMA}·∂u/∂t")


In [ ]:
# ── 5. Initial conditions ────────────────────────────────────────────────────

def _blade(xx, yy, x_center, sign, angle=0.0):
    """
    Single water blade: a sharp ridge of water localised around x_center.

    Shape: Gaussian envelope × smooth Heaviside
      - Gaussian controls blade thickness (sigma = LAMBDA / BLADE_WIDTH_RATIO)
      - Heaviside gives a sharp leading face and smooth trailing tail

    Parameters
    ----------
    xx, yy   : 2D position arrays (indexing='ij')
    x_center : blade center position along the propagation axis (m)
    sign     : +1 (travels in +x) or -1 (travels in -x)
               also controls which face of the blade is sharp
    angle    : tilt of the blade front from the y-axis (radians)
               0.0 → blade parallel to y-axis (straight)
               ALPHA_R → blade tilted by the collision angle
    """
    # Coordinate perpendicular to the blade front (rotated by angle)
    x_perp     = xx * np.cos(angle) + yy * np.sin(angle)
    x_center_p = x_center * np.cos(angle)

    sigma     = LAMBDA / BLADE_WIDTH_RATIO
    gauss     = np.exp(-((x_perp - x_center_p)**2) / (2.0 * sigma**2))
    k_h       = HEAVISIDE_SHARPNESS / sigma

    # Clip sigmoid argument to avoid exp overflow → NaN
    arg       = np.clip(-k_h * sign * (x_center_p - x_perp), -500, 500)
    heaviside = 1.0 / (1.0 + np.exp(arg))
    return np.nan_to_num(gauss * heaviside, nan=0.0)

def initial_condition_b(xx, yy):
    # If the solver is passing (X, Y) in 'ij' format, but your formulas 
    # expect 'xy' format, swapping the inputs corrects the matrix mapping!
    blade1 = _blade(yy, xx, X1, sign=-1, angle=BLADE1_ANGLE)
    blade2 = _blade(yy, xx, X2, sign=+1, angle=BLADE2_ANGLE)
    return blade1 + blade2

def initial_velocity_b(xx, yy):
    # Do the same swap here so the velocity fields align perfectly
    dx = Lx / (xx.shape[1] - 1)
    dy = Ly / (yy.shape[0] - 1)

    b1      = _blade(yy, xx, X1, sign=-1, angle=BLADE1_ANGLE)
    db1_dx  = np.gradient(b1, dx, axis=1)
    vel_b1  = np.nan_to_num(db1_dx * SPEED_FACTOR, nan=0.0)

    b2      = _blade(yy, xx, X2, sign=+1, angle=BLADE2_ANGLE)
    db2_dx  = np.gradient(b2, dx, axis=1)
    db2_dy  = np.gradient(b2, dy, axis=0)
    db2_dir = np.cos(BLADE2_ANGLE) * db2_dx + np.sin(BLADE2_ANGLE) * db2_dy
    vel_b2  = np.nan_to_num(db2_dir * SPEED_FACTOR, nan=0.0)

    return vel_b1 - VELOCITY_SCALE_BLADE2 * vel_b2


In [ ]:
# ── 5b. Plot initial conditions (FIXED) ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Calculate base arrays
ic = initial_condition_b(xx, yy)
iv = initial_velocity_b(xx, yy)

# ── Left: initial elevation ──
# REMOVED .T so that axis 0 (y) is vertical and axis 1 (x) is horizontal
im = axes[0].imshow(ic, origin='lower',
                    extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                    cmap='RdBu_r', aspect='auto')
axes[0].set_title('Initial elevation  η(x,y,0)')
axes[0].set_xlabel('x — cross-shore (m)')
axes[0].set_ylabel('y — longshore (m)')
plt.colorbar(im, ax=axes[0], label='η (m)')


# ── Right: True Physical Velocity Vector Field ──
# Instead of taking the gradient of dη/dt, we construct the true physical WKB 
# propagation direction vectors for the two blades.

# ── Right: True Physical Velocity Vector Field (FIXED) ───────────────────────

# Blade 1 direction: purely in -x direction
b1 = _blade(xx, yy, X1, sign=-1, angle=BLADE1_ANGLE)
vx_b1 = -1.0 * b1
vy_b1 = 0.0 * b1

# Blade 2 direction: normal to the tilted front, traveling right
b2 = _blade(xx, yy, X2, sign=+1, angle=BLADE2_ANGLE)
vx_b2 = np.cos(BLADE2_ANGLE) * b2
vy_b2 = np.sin(BLADE2_ANGLE) * b2

# Combine them to match the velocity field scaling orientation
true_vx = vx_b1 - VELOCITY_SCALE_BLADE2 * vx_b2
true_vy = vy_b1 - VELOCITY_SCALE_BLADE2 * vy_b2

# Subsample the grid for clean quiver plotting
step = Nx // 24
xs   = xx[::step, ::step]
ys   = yy[::step, ::step]
u_q  = true_vx[::step, ::step]
v_q  = true_vy[::step, ::step]

# --- CLEAN SCALING TRICK ---
# Normalize the vectors so they have a consistent, clean length where they exist
magnitude = np.sqrt(u_q**2 + v_q**2)
# Avoid division by zero
magnitude[magnitude == 0] = 1.0
u_q_norm = u_q / magnitude
v_q_norm = v_q / magnitude

# Mask out areas where there is no wave, so we don't draw tiny dot-arrows
mask = (b1[::step, ::step] > 0.05) | (b2[::step, ::step] > 0.05)
u_q_norm[~mask] = 0
v_q_norm[~mask] = 0
# ----------------------------

# Background: scalar velocity magnitude
axes[1].imshow(iv, origin='lower',
               extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
               cmap='RdBu_r', aspect='auto', alpha=0.6)

# Overlay: quiver arrows with explicit coordinate mapping and scale
axes[1].quiver(xs, ys, u_q_norm, v_q_norm,
               angles='xy', scale_units='xy', scale=4.0,
               color='white', alpha=0.9, width=0.004)

axes[1].set_title('Initial wave propagation velocity field')
axes[1].set_xlabel('x — cross-shore (m)')
axes[1].set_ylabel('y — longshore (m)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Solver setup ──────────────────────────────────────────────────────────

solver = PDESolver(equation_num)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition_b,
    initial_velocity=initial_velocity_b,
    n_frames=n_frames,
    plot=True,
)


In [ ]:
# ── 7. Solve ─────────────────────────────────────────────────────────────────

frames = solver.solve()


In [ ]:
# ── 8. Visualization ─────────────────────────────────────────────────────────

# Raise the animation size limit to allow large frame counts at high resolution
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None,
    mode='imshow',
)

HTML(ani.to_jshtml())
